# 06 — The Fisher–Rao Metric

Answer the question `02/05` left open — which ruler is natural on the simplex? — with a criterion, not a preference: two distributions are close when *samples cannot tell them apart*. Make that precise by hand for a coin (the length of a step is the number of standard errors it moves the sample mean), derive the Fisher information $I(\theta) = 1/\theta(1-\theta)$ twice — from the score, and as the Hessian of relative entropy — and see it diverge at the boundary: every step near a corner is expensive (the corners themselves stay at a finite distance — `02/07` computes it — because the blow-up is only like $1/\sqrt p$). Extend it to three outcomes, $ds^2 = \sum_i dp_i^2/p_i$, draw the *unit circles* of this metric across the triangle and watch them squeeze against the edges, and finally meet the theorem that makes it *the* choice: Chentsov's — the only metric that does not change when you split an outcome into sub-outcomes and never grows when you merge outcomes. The geodesics and the closed-form distance are `02/07`'s.

**Prerequisites:** `02/05_the_probability_simplex` (the triangle, barycentric coordinates, the four rulers and their disagreement near the boundary), `02/02_kl_divergence` (relative entropy, its Hessian at $q = p$, monotonicity under coarse-graining), `01/10_expectation_values` (shot noise: the standard error $\sqrt{p(1-p)/N}$ of an estimated probability), `01/14` (the Bhattacharyya overlap behind the Hellinger distance).

**What you'll be able to do**
- Turn "samples cannot tell $\theta$ from $\theta + \delta$" into a length: $\delta$ divided by the standard error $\sqrt{\theta(1-\theta)/N}$ — and read off $I(\theta) = 1/\theta(1-\theta)$ as the metric of a coin, derived by hand from the score $\partial_\theta\log p$.
- Derive $D(p_\theta\|p_{\theta+\delta}) \approx \tfrac12 I(\theta)\delta^2$ (nats) and $\mathrm{Hel}^2 \approx \tfrac18 I\delta^2$ by Taylor expansion — the two rulers of `02/05` that counted surprise are the same ruler up to second order — and check both numerically.
- Write the metric on three outcomes, $g = \mathrm{diag}(1/p_i)$ on tangent vectors with $\sum_i dp_i = 0$, compute the Fisher length of `02/05`'s two $0.02$-shifts, and draw the metric's unit circles (Tissot ellipses) across the triangle.
- State Chentsov's theorem and *check its two halves by hand*: splitting an outcome in fixed proportions leaves every Fisher length unchanged (Euclidean length changes), and merging outcomes never increases it, with equality exactly when the merged ratios agree — the same condition as `02/02`'s log-sum equality.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.geometry.info_geometry import bernoulli_fisher, multinomial_fisher
from qot_course.infotheory.classical import kl_divergence

rng = np.random.default_rng(0)   # coin samples for the distinguishability experiment of section 1
viz.use_course_style()

## 1. Distinguishability, by hand: a step counted in standard errors

Take a coin with $P(\text{heads}) = \theta$ and a second coin with $\theta + \delta$. How many flips do you need to tell them apart? From $N$ flips you estimate $\hat\theta$ with standard error $\sigma_N = \sqrt{\theta(1-\theta)/N}$ (`01/10`). The two coins are distinguishable when the step $\delta$ is a few standard errors wide, $\delta \gtrsim \sigma_N$ — so the natural *length* of the step is not $\delta$ but

$$
\frac{\delta}{\sigma_N} = \delta\,\sqrt{\frac{N}{\theta(1-\theta)}} = \sqrt{N}\;\sqrt{I(\theta)}\;\delta, \qquad I(\theta) = \frac{1}{\theta(1-\theta)} .
$$

The same $\delta$ is a short step near $\theta = \tfrac12$ and a long one near $\theta = 0$ or $1$: a $0.01$ change is invisible on a fair coin but glaring on a coin that almost never lands heads. Equivalently, the number of flips needed to resolve the step at a fixed confidence is $N \approx z^2/(I(\theta)\delta^2)$: **the metric is the price, in samples, of a unit of Euclidean distance**, and $\sqrt{I}\,\delta$ is the step measured in units the data can feel.

**The same number from the score.** Fisher's definition (1925; the object Rao turned into a metric in 1945) is the variance of the *score*, the sensitivity of the log-likelihood to the parameter:
$$
I(\theta) = \mathbb E_{p_\theta}\!\Big[\Big(\frac{\partial\log p_\theta(x)}{\partial\theta}\Big)^2\Big] .
$$
For the coin, $\log p_\theta(\text{heads}) = \log\theta$ and $\log p_\theta(\text{tails}) = \log(1-\theta)$, so the score is $1/\theta$ on heads and $-1/(1-\theta)$ on tails, and its mean square is
$$
\theta\cdot\frac{1}{\theta^2} + (1-\theta)\cdot\frac{1}{(1-\theta)^2} = \frac1\theta + \frac1{1-\theta} = \frac{1}{\theta(1-\theta)} .
$$
(Its mean is $\theta\cdot\tfrac1\theta - (1-\theta)\cdot\tfrac1{1-\theta} = 0$, always: the score has zero mean because $\sum_x p_\theta(x) = 1$ for every $\theta$. That fact will do work in section 2.) Let's measure distinguishability directly — simulate $N$ flips of each of two coins a step $\delta = 0.01$ apart, at $\theta = 0.5$ and at $\theta = 0.97$, and count how many standard errors (of one coin's mean) separate the two sample means — then draw $I(\theta)$.


In [ ]:
def separation_in_se(theta, delta, N, trials=4000):
    # sample means of two coins a step delta apart; how many standard errors apart are they?
    a = rng.binomial(N, theta, size=trials) / N
    b = rng.binomial(N, theta + delta, size=trials) / N
    return (b.mean() - a.mean()) / a.std(ddof=1)

delta, N = 0.01, 2000
for theta in [0.5, 0.8, 0.97]:
    sep = separation_in_se(theta, delta, N)
    print(f"theta = {theta:.2f}: the step {delta} is {sep:5.2f} standard errors wide at N = {N}   (prediction sqrt(N I) delta = {np.sqrt(N * bernoulli_fisher(theta)) * delta:5.2f};  I = {bernoulli_fisher(theta):6.2f})")
for theta in [0.5, 0.97]:
    N_needed = 4 / (bernoulli_fisher(theta) * delta**2)       # z = 2 standard errors
    print(f"theta = {theta:.2f}: flips needed to make the step {delta} two standard errors wide: {N_needed:8.0f}")

thetas = np.linspace(0.005, 0.995, 400)
fisher = np.array([bernoulli_fisher(t) for t in thetas])
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), gridspec_kw={"wspace": 0.28})
ax = axes[0]
ax.plot(thetas, fisher, color=viz.SOURCE_COLOR, lw=2.4)
ax.axvline(0.5, color=viz.TARGET_COLOR, ls="--", lw=1, label="theta = 1/2: I = 4, the flattest point")
ax.set_yscale("log"); ax.set_xlabel("coin parameter theta"); ax.set_ylabel("Fisher information I(theta)  (log scale)")
ax.set_title("I(theta) = 1 / theta(1 - theta)", pad=10); ax.legend(loc="upper center")
ax = axes[1]
ax.plot(thetas, np.sqrt(fisher) * delta, color=viz.SOURCE_COLOR, lw=2.4, label=f"Fisher length of a step {delta}:  sqrt(I) delta")
ax.axhline(delta, color=COLORS["muted"], ls=":", lw=1.2, label=f"its Euclidean length {delta} (arbitrary scale, for reference)")
ax.set_yscale("log"); ax.set_xlabel("coin parameter theta"); ax.set_ylabel("length of the same step  (log scale)")
ax.set_title("Fisher length of one Euclidean step of 0.01", pad=10); ax.legend(loc="upper center")
plt.show()

**Read the output and the figure.** A step of $0.01$ on a fair coin is $0.9$ standard errors wide at $N = 2000$ — invisible; the same step at $\theta = 0.97$ is $2.6$ standard errors wide — a clear separation — and the measured separations match $\sqrt{N I(\theta)}\,\delta$. To make the step two standard errors wide you need about $10{,}000$ flips at $\theta = \tfrac12$ and about $1{,}200$ at $\theta = 0.97$: the metric is the price in samples, and it is eight-to-nine times cheaper near the edge. Left: $I(\theta)$ is smallest at the fair coin ($I = 4$) and climbs without bound at both ends — on a log axis, straight up. Right: the same Euclidean step of $0.01$ has a Fisher length that grows without bound toward the boundary — near the boundary every step is expensive in this metric, not because the probabilities are far, but because a tiny change of a near-zero probability is a large statistical event. (Expensive, not unreachable. What you have built is a **metric**: a rule giving the squared length $ds^2 = I(\theta)\,d\theta^2$ of every infinitesimal step at every point; the length of a *path* is $\int ds$. Here $ds = d\theta/\sqrt{\theta(1-\theta)}$ blows up only like $1/\sqrt\theta$ at the corner, so the integral converges — the corner sits at a finite distance, which `02/07` names.) This is what the Hellinger and KL level sets of `02/05` were flattening against.


## 2. The same metric, twice more: the Hessian of relative entropy, and Hellinger

`02/05` found that KL and Hellinger agreed near a point and disagreed from Euclidean and TV — "statistical rulers count surprise". Here is why: **both are the Fisher metric to second order.** Expand the relative entropy (in nats) of a distribution $p$ to a nearby $p + dp$ (with $\sum_i dp_i = 0$, so the sum stays $1$):
$$
D(p\,\|\,p + dp) = -\sum_i p_i\ln\Big(1 + \frac{dp_i}{p_i}\Big) = -\sum_i p_i\Big(\frac{dp_i}{p_i} - \frac{dp_i^2}{2p_i^2} + \dots\Big) = -\underbrace{\sum_i dp_i}_{0} + \frac12\sum_i\frac{dp_i^2}{p_i} + O(dp^3) .
$$
The first-order term vanishes exactly because the total probability is conserved — the same "zero-mean score" fact as in section 1 — and what is left is *half the Fisher quadratic form*: $D(p\|p + dp) \approx \tfrac12\,ds^2$ with $ds^2 = \sum_i dp_i^2/p_i$. For a coin, $ds^2 = d\theta^2/\theta + d\theta^2/(1-\theta) = I(\theta)\,d\theta^2$ — the same $I$. (The expansion of $D(p + dp\|p)$ gives the same leading term: the asymmetry of $D$ is third order.) And Hellinger: $\mathrm{Hel}^2 = 1 - \sum_i\sqrt{p_i(p_i + dp_i)} = 1 - \sum_i p_i\sqrt{1 + dp_i/p_i} = 1 - \sum_i p_i\big(1 + \tfrac{dp_i}{2p_i} - \tfrac{dp_i^2}{8p_i^2} + \dots\big) = \tfrac18\,ds^2 + O(dp^3)$. Relative entropy, Hellinger, and the sample-counting argument of section 1 are three faces of one quadratic form. Let's check both expansions on a coin and on a three-outcome $p$, for steps shrinking to zero, and watch the ratio to $ds^2$ settle on $\tfrac12$ and $\tfrac18$.


In [ ]:
def fisher_ds2(p, dp):
    return float(np.sum(np.asarray(dp) ** 2 / np.asarray(p)))

def hellinger2(p, q):
    return float(1.0 - np.sum(np.sqrt(np.asarray(p) * np.asarray(q))))

p3 = np.array([0.7, 0.2, 0.1]); v3 = np.array([0.5, -0.2, -0.3])     # a tangent direction: entries sum to 0
print(f"{'step h':>8s} {'D(p||p+h v)/ds^2 (coin, theta=0.3)':>36s} {'Hel^2/ds^2 (coin)':>19s} {'D/ds^2 (3 outcomes)':>21s} {'Hel^2/ds^2 (3 out.)':>20s}")
for hstep in [0.1, 0.03, 0.01, 0.003, 0.001]:
    theta = 0.3; pc = np.array([theta, 1 - theta]); vc = np.array([1.0, -1.0])
    D_c = kl_divergence(pc, pc + hstep * vc, base=np.e); ds2_c = fisher_ds2(pc, hstep * vc)
    D_3 = kl_divergence(p3, p3 + hstep * v3, base=np.e); ds2_3 = fisher_ds2(p3, hstep * v3)
    print(f"{hstep:8.3f} {D_c / ds2_c:36.4f} {hellinger2(pc, pc + hstep * vc) / ds2_c:19.4f} {D_3 / ds2_3:21.4f} {hellinger2(p3, p3 + hstep * v3) / ds2_3:20.4f}")
print(f"limits: 1/2 = {0.5}, 1/8 = {0.125};  and for the coin ds^2 = I(theta) dtheta^2: {fisher_ds2(pc, 0.01 * vc):.6f} = {bernoulli_fisher(0.3) * 0.01**2:.6f}")
# the asymmetry is third order: D(p||q) - D(q||p) shrinks like h^3
for hstep in [0.1, 0.01, 0.001]:
    q = p3 + hstep * v3
    print(f"h = {hstep}: D(p||q) - D(q||p) = {kl_divergence(p3, q, base=np.e) - kl_divergence(q, p3, base=np.e):.2e}  (ratio to h^3: {(kl_divergence(p3, q, base=np.e) - kl_divergence(q, p3, base=np.e)) / hstep**3:.3f})")

**Read the output.** As the step shrinks, $D/ds^2$ settles on $0.5$ and $\mathrm{Hel}^2/ds^2$ on $0.125$, on the coin and on three outcomes alike — the Taylor expansions, confirmed to within $10^{-3}$ at $h = 0.001$ — and the coin's $ds^2$ is exactly $I(\theta)\,d\theta^2$. The asymmetry of relative entropy shrinks like $h^3$ (its ratio to $h^3$ settles on a constant as $h \to 0$), so near a point $D$ *is* symmetric and quadratic: relative entropy is the Fisher metric plus corrections, which is why `02/02`'s two landscapes looked alike near their shared zero. Euclidean distance has the quadratic form $\sum_i dp_i^2$, blind to $p$, and total variation is not even quadratic — it is $\tfrac12\|dp\|_1$, polyhedral, the hexagons of `02/05` — neither comes from a metric that counts samples, and that is the whole difference between the two families of rulers in `02/05`.


## 3. Three outcomes: the metric on the triangle, and its unit circles

For a categorical distribution with $n$ outcomes, parametrised by its own entries, the score for outcome $x$ with respect to $p_j$ is $\partial\log p_x/\partial p_j = \delta_{xj}/p_j$, so
$$
g_{jk} = \sum_x p_x\,\frac{\delta_{xj}}{p_j}\frac{\delta_{xk}}{p_k} = \frac{\delta_{jk}}{p_j}, \qquad g = \mathrm{diag}\Big(\frac1{p_1}, \dots, \frac1{p_n}\Big), \qquad ds^2 = \sum_i \frac{dp_i^2}{p_i} ,
$$
to be applied to *tangent* vectors $dp$ with $\sum_i dp_i = 0$ — the directions that stay on the simplex (the helper `multinomial_fisher` returns this diagonal matrix; the constraint is yours to respect). Strictly, $n$ free entries are one too many for a statistical model; the honest statement is that in *any* parametrisation $\theta$ the score variance is $I_{ab} = \sum_x \partial_a p_x\,\partial_b p_x/p_x$, so the length of a tangent step is always $\sum_x dp_x^2/p_x$ — in the reduced coordinates $(p_1, p_2)$ the matrix is not diagonal, but it is the same quadratic form, and "$\mathrm{diag}(1/p)$ restricted to the tangent plane" is the coordinate-free way to write it. The metric is **diagonal but not constant**: the price of moving outcome $i$ is $1/p_i$, steep where $p_i$ is small. Take `02/05`'s two moves of $0.02$ of mass from $p = (0.5, 0.48, 0.02)$: between the two common outcomes, $ds^2 = 0.02^2\,(1/0.5 + 1/0.48) = 0.0016$; taking the rare outcome's $0.02$ away, $ds^2 = 0.02^2\,(1/0.02 + 1/0.5) = 0.0208$ — about thirteen times larger for the same Euclidean step. One honesty note: $ds^2$ is the *infinitesimal* price at $p$, and this second move removes *all* of outcome $3$ — a $100\%$ relative change, far outside the linear regime — so the tangent price undershoots the true length: the exact Fisher–Rao lengths (`02/07`'s closed form) are $0.040$ and $0.285$, a ratio of $7$ — the same $7$ that `02/05`'s Hellinger found, as section 2 predicts. Here we compare the *tangent* prices, $0.0016$ vs $0.0208$; a step that crossed the edge would have infinite tangent price.

The cleanest picture of a metric is its **unit circle** at each point: the set of tangent vectors of Fisher length $r$ (a small fixed $r$), drawn on the triangle. Where the metric is large the circle is small, and where it is larger in one direction than another the circle is an ellipse (cartographers call these Tissot's indicatrices). Let's compute them by hand: choose an orthonormal basis $e_1 = (1,-1,0)/\sqrt2$, $e_2 = (1,1,-2)/\sqrt6$ of the tangent plane (two unit vectors, orthogonal to each other and to $(1,1,1)$ — so their combinations are exactly the $dp$ with $\sum_i dp_i = 0$), write the $2\times2$ matrix $G = E^\top\mathrm{diag}(1/p)E$, and draw $\{v : v^\top G v = r^2\}$ at a grid of points. $G$ is symmetric and positive, so in its eigenbasis $v^\top G v = \lambda_1 v_1^2 + \lambda_2 v_2^2$ (eigenvalues of a Hermitian matrix, `01/12`): the set is an ellipse with semi-axes $r/\sqrt{\lambda_k}$ along the eigenvectors — large eigenvalue, short axis.


In [ ]:
V = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])      # the triangle of 02/05
E = np.stack([np.array([1, -1, 0]) / np.sqrt(2), np.array([1, 1, -2]) / np.sqrt(6)], axis=1)   # orthonormal basis of the tangent plane (3 x 2)

def tissot(p, r=0.1, m=80):
    G = E.T @ multinomial_fisher(p) @ E                        # the 2x2 Fisher metric in tangent coordinates
    w, U = np.linalg.eigh(G)
    ang = np.linspace(0, 2 * np.pi, m)
    circle = np.stack([np.cos(ang), np.sin(ang)])             # unit circle in the eigenbasis ...
    v = U @ (circle / np.sqrt(w)[:, None]) * r                 # ... scaled to Fisher length r
    dp = E @ v                                                 # back to 3-D tangent vectors (sum to 0)
    return (p[None, :] + dp.T) @ V, w                          # ellipse points in the plane, and the two eigenvalues

p_edge = np.array([0.5, 0.48, 0.02])
for name, dp in [("between the common outcomes", np.array([-0.02, 0.02, 0.0])), ("remove the rare outcome's mass", np.array([0.0, 0.02, -0.02]))]:
    print(f"{name:32s}: Euclidean^2 = {np.sum(dp**2):.4f}   Fisher ds^2 = {fisher_ds2(p_edge, dp):.4f}   ratio ds^2/Euclid^2 = {fisher_ds2(p_edge, dp) / np.sum(dp**2):6.1f}")

fig, ax = plt.subplots(figsize=(8.5, 7.2))
tri = np.vstack([V, V[:1]]); ax.plot(tri[:, 0], tri[:, 1], color=COLORS["muted"], lw=1.5)   # the triangle of 02/05, by hand
for (x, y), lab, dx, dy in zip(V, [r"$\delta_1=(1,0,0)$", r"$\delta_2=(0,1,0)$", r"$\delta_3=(0,0,1)$"], [-0.07, 0.07, 0.0], [-0.045, -0.045, 0.045]):
    ax.text(x + dx, y + dy, lab, ha="center", va="center", fontsize=11, color=COLORS["text"])
ax.set_aspect("equal"); ax.axis("off")
grid_pts = []
for i in range(1, 11):
    for j in range(1, 11 - i):
        a_, b_ = i / 11, j / 11
        grid_pts.append(np.array([a_, b_, 1 - a_ - b_]))
for p in grid_pts:
    pts, w = tissot(p)
    ax.plot(pts[:, 0], pts[:, 1], color=viz.SOURCE_COLOR, lw=1.6)
    ax.scatter(*(p @ V), s=6, color=COLORS["text"], zorder=4)
ax.set_title("Unit circles of the Fisher metric (equal Fisher length r = 0.1) across the triangle", pad=26)
plt.show()
w_centre = tissot(np.array([1/3, 1/3, 1/3]))[1]; w_edge = tissot(np.array([0.45, 0.45, 0.10]))[1]
print(f"metric eigenvalues at the centre: {np.round(w_centre, 2)} (isotropic, = 3 = 1/p);  near the bottom edge (0.45, 0.45, 0.10): {np.round(w_edge, 2)} -- {w_edge.max() / w_edge.min():.1f} x stiffer across the edge than along it")

**Read the output and the figure.** The two $0.02$-moves of `02/05` have the same Euclidean length and tangent Fisher prices in the ratio $13$ (exact lengths: ratio $7$, the Hellinger number of `02/05` — the linearisation is only a guide for a move that empties an outcome): the metric charges $1/p_i$ per unit of squared displacement of outcome $i$, and the rare outcome is expensive. On the triangle, the unit circles are round and largest at the centre, where $g = 3I_2$ and every direction costs the same, and they shrink and flatten into ellipses toward the edges — squeezed *across* the nearby edge (moving the small probability) and relatively free *along* it (trading mass between the two large probabilities): near $(0.45, 0.45, 0.10)$ the metric is $3.3$ times stiffer across than along. The corners, where two probabilities vanish, are stiff in every direction — the ellipses would shrink to points there, although, as the coin computation showed, the total length to a corner stays finite. The picture is the answer to `02/05`'s question in one figure — the triangle is not flat, and the shortest paths of `02/07` will bend away from the boundary exactly because the metric is cheaper in the middle.


## 4. Why this metric: Chentsov's theorem, checked by hand

There are many Riemannian metrics on the triangle. What singles this one out is a criterion, not a taste: a statistical distance should not depend on *how finely you happen to label the outcomes*. Two operations make that precise (a third, *relabelling* — permuting the outcomes — is free: $\sum_i dp_i^2/p_i$ is a sum over outcomes and does not care about their order).

- **Splitting** (a Markov embedding): replace outcome $1$ by two sub-outcomes in fixed proportions $\lambda, 1 - \lambda$, so $p = (p_1, p_2, p_3) \mapsto p' = (\lambda p_1, (1-\lambda)p_1, p_2, p_3)$ and every tangent vector $dp \mapsto dp' = (\lambda\,dp_1, (1-\lambda)\,dp_1, dp_2, dp_3)$. The finer description carries no information about $p$ that the coarse one did not (the split is a fixed coin that knows nothing about $p$: the coarse description already carries all the information — a *sufficient statistic*), so a statistical metric must give $dp'$ the same length as $dp$. Check by hand: $\frac{(\lambda dp_1)^2}{\lambda p_1} + \frac{((1-\lambda)dp_1)^2}{(1-\lambda)p_1} = \frac{dp_1^2}{p_1}(\lambda + 1 - \lambda) = \frac{dp_1^2}{p_1}$ — **exactly invariant**, for every $\lambda$. The Euclidean length is not: $(\lambda dp_1)^2 + ((1-\lambda)dp_1)^2 < dp_1^2$.
- **Merging** (coarse-graining, `02/02` §4(c)): lump outcomes $1$ and $2$ into one, $p \mapsto (p_1 + p_2, p_3)$, $dp \mapsto (dp_1 + dp_2, dp_3)$. Information can be lost, never gained, so the length must not increase: $\frac{(dp_1 + dp_2)^2}{p_1 + p_2} \le \frac{dp_1^2}{p_1} + \frac{dp_2^2}{p_2}$ — the inequality $(a + b)^2 \le (\frac{a^2}{s} + \frac{b^2}{t})(s + t)$ with $a = dp_1, b = dp_2, s = p_1, t = p_2$, which is $(a\sqrt{t/s} - b\sqrt{s/t})^2 \ge 0$ expanded (Cauchy–Schwarz) — with equality iff $dp_1/p_1 = dp_2/p_2$: merging costs nothing exactly when the two outcomes were being moved *in proportion* — the same condition as the equality case of `02/02`'s log-sum inequality, which is no coincidence (section 2: $D \approx \tfrac12 ds^2$).

**Chentsov's theorem** (1972; short proof and extension to unnormalised measures, Campbell 1986; general sample spaces, Ay, Jost, Lê & Schwachhöfer 2015): *the Fisher metric is, up to a constant factor, the only Riemannian metric — defined on the simplices of all sizes at once — that is invariant under splitting;* monotonicity under merging is the complementary property checked above. The ruler that counts samples is forced. Let's verify both halves numerically on random points and directions — and watch the Euclidean length fail the first one.


In [ ]:
def split_outcome(p, dp, lam):
    return np.concatenate([[lam * p[0], (1 - lam) * p[0]], p[1:]]), np.concatenate([[lam * dp[0], (1 - lam) * dp[0]], dp[1:]])

def merge_first_two(p, dp):
    return np.concatenate([[p[0] + p[1]], p[2:]]), np.concatenate([[dp[0] + dp[1]], dp[2:]])

worst_split, worst_merge = 0.0, -np.inf
for _ in range(2000):
    p = rng.dirichlet(np.ones(3)); dp = rng.normal(size=3); dp -= dp.mean(); dp *= 0.01   # a random tangent direction
    lam = rng.uniform(0.05, 0.95)
    ps, dps = split_outcome(p, dp, lam); pm, dpm = merge_first_two(p, dp)
    worst_split = max(worst_split, abs(fisher_ds2(ps, dps) - fisher_ds2(p, dp)))
    worst_merge = max(worst_merge, fisher_ds2(pm, dpm) - fisher_ds2(p, dp))
print(f"splitting outcome 1 in random proportions: max |ds'^2 - ds^2| over 2000 trials = {worst_split:.1e}   (exactly invariant)")
print(f"merging outcomes 1 and 2:                  max (ds'^2 - ds^2) over 2000 trials  = {worst_merge:.2e}   (never positive)")
p = np.array([0.5, 0.3, 0.2]); dp = np.array([0.02, -0.015, -0.005]); ps, dps = split_outcome(p, dp, 0.3)
print(f"one example, p = {p}, dp = {dp}: Fisher ds^2 = {fisher_ds2(p, dp):.6f} -> after a 30/70 split {fisher_ds2(ps, dps):.6f};  Euclidean |dp|^2 = {np.sum(dp**2):.6f} -> {np.sum(dps**2):.6f} (changed)")
dp_prop = np.array([0.02, 0.012, -0.032])        # dp_1/p_1 = dp_2/p_2 = 0.04: moved in proportion
pm, dpm = merge_first_two(p, dp_prop)
print(f"merging two outcomes moved IN PROPORTION (dp_i/p_i equal): ds^2 = {fisher_ds2(p, dp_prop):.6f} -> {fisher_ds2(pm, dpm):.6f}  (no loss: the equality case)")
pm, dpm = merge_first_two(p, dp)
print(f"merging two outcomes moved out of proportion:              ds^2 = {fisher_ds2(p, dp):.6f} -> {fisher_ds2(pm, dpm):.6f}  (length lost)")

**Read the output.** Over two thousand random points, directions and split proportions, the Fisher length is unchanged by splitting to $10^{-16}$ — exact, as the one-line computation said — while the Euclidean length of the same step changes; and merging never increases it. The two merges of the last lines are the equality case and its failure: move two outcomes in proportion and lumping them loses nothing; move them out of proportion and length is lost. A metric that passes both tests is a metric that measures *information about $p$* and nothing about the labelling — and Chentsov's theorem says there is only one. That is the answer to `02/05`: the natural ruler on the simplex is the one that counts samples, it is forced by invariance, and it makes the triangle curved.


## 5. Where the metric goes

- **Geodesics and the closed form** (`02/07`): a metric gives shortest paths and a distance along them. For this metric both have a closed form, reached by a change of variable that turns the triangle into a piece of a sphere — `02/07` does it; today you know *why* the paths will bend away from the edges.
- **Cramér–Rao**: the metric also bounds estimation. For any unbiased estimator $\hat\theta$ from $N$ samples, $\mathrm{Var}(\hat\theta) \ge 1/(N I(\theta))$. For the coin the sample mean has variance $\theta(1-\theta)/N = 1/(N I)$ exactly — it *attains* the bound, which is why `01/10`'s standard error is what it is (exercise 4).
- **The quantum Fisher information** (`02/12`): replace $p$ by a density matrix $\rho$ and ask the same question — how well can measurements tell $\rho$ from $\rho + d\rho$? — and Chentsov's uniqueness *breaks*: there is a whole family of monotone metrics (Petz 1996), and the smallest is the **Bures metric**, which on diagonal states reduces to $\tfrac14\sum_i dp_i^2/p_i$ — today's metric, scaled. The dictionary row below records the promise.
- **A second geometry** (`02/08`, module 03): everything today is *vertical* — it compares masses bin by bin and is invariant under relabelling (Chentsov's operations never look at which bin is next to which). When the bins have a geometry of their own, a second ruler appears, and it measures something this one cannot.


## Your turn

1. **The flattest coin.** Confirm `bernoulli_fisher(0.5) == 4` and that it is the minimum over $\theta$ by plotting $I$ on a linear axis; then explain in one sentence, using section 1, why the *most uncertain* coin is the *least sensitive* to a change of $\theta$. Compute the number of flips needed to resolve $\delta = 0.005$ at two standard errors for $\theta \in \{0.5, 0.9, 0.99\}$.
2. **Stiffest coordinate, and an ellipse of your own.** With `multinomial_fisher([0.7, 0.2, 0.1])`, which coordinate is stiffest? Draw the Fisher unit circle at that point with `tissot` and at the centre, and give the ratio of their areas (the area of the ellipse is $\pi r^2/\sqrt{\det G}$ — check numerically).
3. **The Hessian in the other direction.** Section 2 expanded $D(p\|p + dp)$. Expand $D(p + dp\|p)$ by hand to second order and show the leading term is the same $\tfrac12\sum_i dp_i^2/p_i$; then verify numerically on three outcomes that $\big(D(p\|q) + D(q\|p)\big)/ds^2 \to 1$ and that the *difference* $D(p\|p+dp) - D(p+dp\|p)$ is $-\tfrac16\sum_i dp_i^3/p_i^2$ to leading order — check that the printed ratio $0.44$ in section 2 is exactly that number for `p3, v3`.
4. **Harder: Cramér–Rao, and the bound a biased estimator can beat.** (a) For the coin, estimate $\theta = 0.3$ from $N = 50$ flips with the sample mean, $5000$ times, and compare its variance with $1/(N I(\theta))$. (b) Now use the shrinkage estimator $\hat\theta' = (k + 1)/(N + 2)$ ($k$ heads; the posterior mean of a uniform prior): compute its variance *and* its bias; show the variance is below the Cramér–Rao bound and say why that is allowed (state the biased Cramér–Rao bound $\mathrm{Var} \ge (1 + b'(\theta))^2/(N I)$ and check it). (c) (independent of (a)–(b)) Prove the merging inequality of section 4 from Cauchy–Schwarz for $m$ outcomes merged into one, and show that the chain of two merges is again a merge — so the Fisher length is monotone under *every* coarse-graining, not only pairwise ones.


## What you built

- You turned "samples cannot tell them apart" into a length — a step divided by the standard error it moves — and derived $I(\theta) = 1/\theta(1-\theta)$ by hand from the score, then measured it: the same $0.01$ step is $0.9$ standard errors at a fair coin and $2.6$ at $\theta = 0.97$, and the price in flips differs eightfold.
- You derived $D(p\|p + dp) \approx \tfrac12\sum_i dp_i^2/p_i$ and $\mathrm{Hel}^2 \approx \tfrac18\sum_i dp_i^2/p_i$ by Taylor expansion and watched the ratios settle on $\tfrac12$ and $\tfrac18$: the two rulers of `02/05` that counted surprise are the Fisher metric to second order, and the asymmetry of $D$ is third order.
- You wrote the metric on three outcomes, $g = \mathrm{diag}(1/p_i)$ on the tangent plane, priced `02/05`'s two moves ($13\times$ in tangent price, $7\times$ in exact length), and drew its unit circles across the triangle — round at the centre, squeezed against the edges, vanishing at the corners.
- You checked Chentsov's two invariances by hand and numerically — splitting leaves every Fisher length unchanged (Euclidean fails), merging never increases it, equality iff moved in proportion — and stated the theorem that makes this metric the only one.

The triangle is curved, its boundary is expensive to approach, and you know why the ruler that says so is forced. `02/07` walks its shortest paths.


**Dictionary so far** (module 01's table, the four classical-information rows, the simplex row of `02/05`, plus today's row — the geometry arc's first answer; the quantum column is `02/12`'s promise):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| **conditional logic** on two bits: controlled-NOT / XOR, $(a, b) \mapsto (a, a \oplus b)$ — a reversible *permutation* of the four outcomes; on a joint probability table it permutes the four entries; two independent operations, one per bit, can never make the bits agree when they did not before | **CNOT**, the $4\times4$ permutation unitary $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes X$ (controlled-$U$ pattern; control = left factor in this course, Qiskit's `cx(1, 0)`); it permutes the four basis vectors and, by linearity, acts on *superpositions* of the control: $\mathrm{CNOT}\,(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$ — a product in, a non-product out (amplitude-matrix determinant $\ne 0$); no $M \otimes N$ can do that; CNOT$(H \otimes I)|00\rangle$ is the Bell state, whose corners are *coherence* between $|00\rangle$ and $|11\rangle$ (a product $|{+}{+}\rangle$ has them too; the coins have none) — entanglement proper is `01/18` |
| a joint table whose whole is *certain* (one outcome, probability $1$) has certain marginals; mixed marginals force a mixed whole (the coins); the correlation of two bits is fully described by their joint table, and any table is a mixture of point masses — every classical correlation is a *shared randomness* | **entanglement**: a *pure* joint state that is not a product (amplitude-matrix $\det\Psi \ne 0$) has **mixed parts** — pure whole, mixed parts, impossible for any table; measured by the **entanglement entropy** $S(\rho_A) = -\sum_i \lambda_i\log_2\lambda_i$ (eigenvalues of the part; $= S(\rho_B)$; $1$ bit for a Bell state; invariant under local gates $U\otimes V$) — **valid only when the whole is pure**: the coins have $S(\rho_A) = 1$ bit and no entanglement; a mixed state is **separable** if it is a mixture of products, and for two qubits the **partial transpose** decides it (a negative eigenvalue = entangled: Bell $-\tfrac12$, coins none); Bell and coins share every $Z$-statistic and differ in $X$: $\langle X\otimes X\rangle = 1$ vs $0$ |
| the general evolution of a probability vector is a **stochastic matrix** (Markov kernel) $p \to Tp$: columns are probability vectors, so probabilities stay non-negative and sum to $1$; a permutation is the reversible case; noise = a coin you did not see being tossed | a **quantum channel**: the general evolution $\rho \to \sum_k K_k\rho K_k^\dagger$ (Kraus operators, $\sum_k K_k^\dagger K_k = I$) — *derived* as "couple to an environment, evolve unitarily, trace the environment out" ($K_k = \langle k|_E\,U\,|0\rangle_E$); linear, trace-preserving and **completely** positive (the transpose is positive but sends half a Bell pair to a matrix with a $-\tfrac12$ eigenvalue — not a channel); a unitary is the reversible case ($K = U$), a stochastic matrix is the diagonal case ($K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$); **decoherence = entanglement with an environment, then a partial trace**: dephasing (the environment learns the bit; coherence $\times e^{-t/T_\varphi}$), amplitude damping (energy leaks out; population $\times e^{-t/T_1}$, coherence $\times e^{-t/2T_1}$), depolarizing ($\rho \to (1-p)\rho + p\,I/2$); together they give $1/T_2 = 1/2T_1 + 1/T_\varphi$, hence $T_2 \le 2T_1$ — the laws `01/15` fitted |
| **Shannon entropy** $H(p) = -\sum_x p_x\log_2 p_x$: the average *surprise* $-\log_2 p_x$ (the log is forced by additivity of independent surprises); $0$ for a certain outcome, $\log_2 n$ for the uniform (Jensen), concave; **binary entropy** $h(p)$; joint and conditional entropy with the chain rule $H(AB) = H(A) + H(B|A)$, $H(B|A) \ge 0$, hence $H(A) \le H(AB)$ — a part is never more uncertain than the whole (coins: $1 = 1$; independent: $H(AB) = H(A) + H(B)$); the floor of lossless compression (Shannon 1948) | **von Neumann entropy** $S(\rho) = -\mathrm{tr}\,\rho\log_2\rho = H(\text{eigenvalues of }\rho)$ — the Shannon entropy of the eigenvalue coin (`01/13`): $0$ for a pure state, $1$ bit for $I/2$, $h\big(\tfrac{1 + |\vec r|}{2}\big)$ for a qubit; *but* the chain rule's positivity fails: the Bell state has $S(AB) = 0 < S(A) = 1$ (`01/18`) — a negative conditional entropy, `02/11`; the quantum side is earned in `02/09` |
| **relative entropy** (Kullback–Leibler) $D(p\|q) = \sum_x p_x\log_2\frac{p_x}{q_x}$ — the *extra* bits per symbol of coding with model $q$ when the truth is $p$ (cross-entropy minus entropy); **Gibbs**: $D \ge 0$, $= 0$ iff $p = q$ (from $\ln x \le x - 1$) — which re-proves $H \le \log_2 n$ and the source-coding floor; **not** a distance: asymmetric, no triangle inequality, $+\infty$ when $q$ gives zero to something $p$ can do; additive on products, jointly convex, and **monotone under any stochastic map** $D(Tp\|Tq) \le D(p\|q)$ — processing cannot create distinguishability; $D(p\,\|\,\text{uniform}) = \log_2 n - H(p)$ | **Umegaki relative entropy** $S(\rho\|\sigma) = \mathrm{tr}\,\rho(\log_2\rho - \log_2\sigma)$ (`02/09`): equals $D$ when $\rho, \sigma$ are diagonal in one basis, $+\infty$ when the support of $\rho$ leaves that of $\sigma$, $\ge 0$ (Klein), monotone under every quantum channel (Lindblad 1975) — and it *sees the corners*: $|{+}\rangle$ vs $I/2$ have identical diagonals, classical $D = 0$, quantum $S = 1$ bit |
| **mutual information** $I(X;Y) = D(p_{XY}\,\|\,p_X \otimes p_Y) = H(X) + H(Y) - H(XY) = H(X) - H(X|Y)$ — the bits the joint saves over the product, the failure of independence; symmetric, $\ge 0$ (Gibbs), $= 0$ iff independent, sign-blind (anti-correlation counts), sees nonlinear dependence; **ceiling** $I \le \min(H(X), H(Y))$ — two bits share at most $1$ bit; $1 - h(\epsilon)$ through a symmetric flip channel; **conditional** $I(X;Y|Z)$, chain rule $I(X;YZ) = I(X;Y) + I(X;Z|Y)$, hence the **data-processing inequality** $I(X;Z) \le I(X;Y)$ on a chain $X \to Y \to Z$; estimated from counts it is biased *upward* by $\approx \frac{(|X|-1)(|Y|-1)}{2N\ln 2}$ — compare with surrogates | **quantum mutual information** $I(A{:}B) = S(A) + S(B) - S(AB) = S(\rho_{AB}\,\|\,\rho_A \otimes \rho_B)$ (`02/10`): the same formula with von Neumann entropies, $\ge 0$, $= 0$ iff product; *total* correlation, classical and quantum together — the coins give $1$ bit (as here), the Bell state $2$ bits, **above** the classical ceiling, because $S(AB) = 0 < S(A)$; monotone under local channels |
| **transfer entropy** $\mathrm{TE}_{Y\to X} = I(X_{t+1};\,Y_t\,|\,X_t) = H(X_{t+1}|X_t) - H(X_{t+1}|X_t, Y_t)$ — the conditional mutual information of `02/03` with the axes named by time: what the source's past adds to the target's future *beyond the target's own past*; **directed** (lag-1, one symbol of history here; Schreiber 2000); $1 - h(\epsilon)$ for a noisy copy; zero in the back direction where lagged mutual information is fooled by the source's autocorrelation; estimated from counts with the same upward bias, $\approx \frac{|Z|(|X|-1)(|Y|-1)}{2N\ln 2}$, and tested against **autocorrelation-preserving surrogates** (circular shifts), never against a naive shuffle for an unconditioned measure; blind to synergy (XOR), sensitive to the chosen lag | **no quantum lift in this course** — the quantum coupling measures of module 04 ($I(A{:}B)$, Bures, the QOT cost on an embedded pair) are *symmetric*; direction, when you need it, stays with the classical transfer entropy on the raw signals (directed quantum information measures exist in the literature but are not built here) |
| the **probability simplex** $\Delta_{n-1} = \{p \in \mathbb R^n : p_i \ge 0, \sum_i p_i = 1\}$ — the set of *all* distributions on $n$ outcomes, a convex polytope of dimension $n - 1$ (a triangle for $n = 3$): vertices = certain outcomes, edges/faces = a forbidden outcome, centre = uniform; every point is a mixture of the vertices with itself as the weights (barycentric coordinates), mixtures never leave it; entropy is a concave hill on it (peak $\log_2 n$ at the centre, $0$ at the vertices), relative entropy an asymmetric bowl around any point; the product tables of two variables form a curved **independence surface** inside the simplex of joint tables, and mutual information is the relative entropy to it; the *distance* between two points is not given by the set — Euclidean, total variation, Hellinger and KL disagree, most of all near the boundary (`02/06`–`02/08` decide) | the **state space** of a $d$-level quantum system: the density matrices $\rho \ge 0$, $\mathrm{tr}\rho = 1$ — a convex body of dimension $d^2 - 1$ whose *diagonal slice* in any basis is the simplex (`01/12`: a $\rho$ diagonal in a shared basis *is* a $p$); for a qubit, the Bloch ball (`01/13`) — pure states on the surface, mixtures inside, $I/2$ at the centre — and the simplex is one diameter of it; its distances restrict to the simplex's on that slice: trace distance to total variation exactly, Bures (`02/12`) to Hellinger up to a factor $\sqrt2$ |
| the **Fisher information metric** on the simplex, $ds^2 = \sum_i dp_i^2/p_i$ ($g = \mathrm{diag}(1/p_i)$ on tangent vectors with $\sum_i dp_i = 0$; $I(\theta) = 1/\theta(1-\theta)$ for a coin): the length of a step is the number of *standard errors* it moves the sample mean — statistical distinguishability — so the same Euclidean step is longer where probabilities are small — the metric coefficients blow up at the boundary, though the corners stay at finite distance (`02/07`: $\le \pi$); it is the **Hessian of relative entropy** ($D(p\|p + dp) \approx \tfrac12 ds^2$ in nats) and of the Hellinger distance ($\mathrm{Hel}^2 \approx \tfrac18 ds^2$) — the rulers of `02/05` that count surprise agree on it; **Chentsov**: the unique metric (up to scale) unchanged by splitting an outcome in fixed proportions and never increased by merging outcomes (monotone under coarse-graining, equality iff the merged ratios agree); Cramér–Rao: no unbiased estimator beats $1/(N I)$ | the **quantum Fisher information** — not unique: every monotone Riemannian metric on density matrices (Petz's family) lifts Chentsov's criterion, and the smallest of them is the **Bures metric** (`02/12`), $ds^2_{\rm Bures} = \tfrac14\,\mathrm{tr}(d\rho\,L)$ with $d\rho = \tfrac12(\rho L + L\rho)$ — on diagonal states it *is* $\tfrac14\sum_i dp_i^2/p_i$; the quantum Cramér–Rao bound is how a quantum sensor's precision is counted |
| transport plan | quantum coupling |

## What's next

A curved space has curved shortest paths. In `02/07_fisher_rao_geodesics` we find them — and the distance along them — in closed form, through a change of variable that turns the triangle into a piece of a sphere; the geodesics bow away from the boundary, exactly as today's unit circles predict, and the Bhattacharyya overlap of `01/14` turns out to be the cosine of the distance.


## References

- R. A. Fisher, *Theory of statistical estimation*, Math. Proc. Cambridge Philos. Soc. **22**, 700–725 (1925) — the information as the variance of the score. doi:10.1017/S0305004100009580
- C. R. Rao, *Information and accuracy attainable in the estimation of statistical parameters*, Bull. Calcutta Math. Soc. **37**, 81–91 (1945) — the Fisher information as a Riemannian metric, and the Cramér–Rao bound.
- N. N. Chentsov (Čencov), *Statistical Decision Rules and Optimal Inference*, Nauka (1972); Engl. transl. Translations of Mathematical Monographs **53**, AMS (1982) — uniqueness of the Fisher metric under Markov morphisms.
- L. L. Campbell, *An extended Čencov characterization of the information metric*, Proc. Amer. Math. Soc. **98**, 135–141 (1986) — the statement on the simplex used here. doi:10.1090/S0002-9939-1986-0848890-5
- N. Ay, J. Jost, H. V. Lê & L. Schwachhöfer, *Information geometry and sufficient statistics*, Probab. Theory Relat. Fields **162**, 327–364 (2015) — Chentsov's theorem in full generality. doi:10.1007/s00440-014-0574-8
- S.-i. Amari, *Information Geometry and Its Applications*, Springer (2016) — ch. 1 (divergence → metric, $D \approx \tfrac12 ds^2$), ch. 3 (invariance, information monotonicity, Chentsov's theorem). doi:10.1007/978-4-431-55978-8
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., Wiley (2006) — §11.10 (Fisher information and the Cramér–Rao inequality). doi:10.1002/047174882X
- D. Petz, *Monotone metrics on matrix spaces*, Linear Algebra Appl. **244**, 81–96 (1996) — the quantum family of monotone metrics, of which Bures (`02/12`) is the smallest. doi:10.1016/0024-3795(94)00211-8

**Previous:** `notebooks/02_InformationAndGeometry/05_the_probability_simplex.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/07_fisher_rao_geodesics.ipynb`
